In [0]:
%run ./_local_config

### Read bronze

In [0]:
import pandas as pd
import json
from azure.storage.blob import BlobServiceClient

conn_str = f"DefaultEndpointsProtocol=https;AccountName={storage_account_name};AccountKey={storage_account_key};EndpointSuffix=core.windows.net"

blob_service = BlobServiceClient.from_connection_string(conn_str)
blob_client = blob_service.get_blob_client(container="bronze", blob="erp/battery/battery.json")

stream = blob_client.download_blob().readall()
text = stream.decode("utf-8-sig")
pages = [json.loads(line) for line in text.splitlines() if line.strip()]

all_records = []
for page in pages:
    all_records.extend(page["value"])

bronze = pd.DataFrame(all_records)
print(bronze.shape)
bronze.head()

### Clean to silver

In [0]:
import sys
sys.path.append("/Workspace/Users/venura-it@brownsgroup.com/Exide sales/Exide-Sales-Forecast")

from src.transform.clean_silver import clean_to_silver

silver = clean_to_silver(bronze)
print(silver.shape)
silver.head()

### Sanity checks

In [0]:
print(silver["documentType"].value_counts())
print(silver.groupby("documentType")["net_units"].describe())
print(f"Total profit: {silver['profit'].sum():,.2f}")

### Vehicle type

In [0]:
unique_vehicle_types = sorted(silver["vehicle_type"].dropna().unique())
vehicle_type_lookup = pd.DataFrame({
    "vehicle_type_id": [f"V{i}" for i in range(1, len(unique_vehicle_types) + 1)],
    "vehicle_type": unique_vehicle_types
})
print(vehicle_type_lookup)

### Brand counts

In [0]:
brand_counts = silver.groupby(["brand_code", "brand_description"]).size().reset_index(name="count")
print(brand_counts.sort_values("count", ascending=False).to_string())

### Save silver

In [0]:
silver_json = silver.to_json(orient="records", date_format="iso", lines=False)
blob_client = blob_service.get_blob_client(container="silver", blob="erp/battery/battery_clean.json")
blob_client.upload_blob(silver_json, overwrite=True)
print(f"Saved {len(silver)} rows to silver/erp/battery/battery_clean.json")

### Check data reliability

In [0]:
recent_90 = silver[silver["posting_date"] >= silver["posting_date"].max() - pd.Timedelta(days=90)]
daily_counts_90 = recent_90.groupby(recent_90["posting_date"].dt.date).size()
print(daily_counts_90.tail(30))

### Trim to reliable data

In [0]:
cutoff_date = pd.Timestamp("2026-06-12")
silver_trimmed = silver[silver["posting_date"] <= cutoff_date].copy()
print(f"Original: {len(silver)}, Trimmed: {len(silver_trimmed)}")

### Build weekly gold

In [0]:
from src.transform.build_gold_features import build_gold_overall_weekly, build_gold_overall_monthly

gold_weekly = build_gold_overall_weekly(silver_trimmed)
print(gold_weekly.shape)
gold_weekly.tail(10)

###  Build monthly gold

In [0]:
monthly_cutoff = pd.Timestamp("2026-05-31")
silver_trimmed_monthly = silver[silver["posting_date"] <= monthly_cutoff].copy()

gold_monthly = build_gold_overall_monthly(silver_trimmed_monthly)
print(gold_monthly.tail(5)[["month_start", "total_units_sold"]])

In [0]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 1, figsize=(16, 8))

axes[0].plot(gold_weekly["week_start"], gold_weekly["total_units_sold"])
axes[0].set_title("Weekly Net Battery Sales (3 Brands, Trimmed)")
axes[0].set_ylabel("Net Units")

axes[1].plot(gold_monthly["month_start"], gold_monthly["total_units_sold"], marker="o")
axes[1].set_title("Monthly Net Battery Sales (3 Brands, Trimmed)")
axes[1].set_ylabel("Net Units")

plt.tight_layout()
plt.show()

### Save both gold datasets

In [0]:
import io

for name, df in [("weekly", gold_weekly), ("monthly", gold_monthly)]:
    buffer = io.BytesIO()
    df.to_parquet(buffer, index=False)
    buffer.seek(0)
    blob_client = blob_service.get_blob_client(container="gold", blob=f"erp/battery/phase1_overall_{name}.parquet")
    blob_client.upload_blob(buffer, overwrite=True)
    print(f"Saved {len(df)} rows to gold/erp/battery/phase1_overall_{name}.parquet")

### Weekly model: train/test split

In [0]:
feature_cols_weekly = ["week_of_year", "month", "contains_month_end", "lag_4w", "rolling_avg_4w"]
target_col = "total_units_sold"

model_data_weekly = gold_weekly.dropna(subset=feature_cols_weekly + [target_col]).copy()
split_idx = int(len(model_data_weekly) * 0.8)
train_weekly = model_data_weekly.iloc[:split_idx]
test_weekly = model_data_weekly.iloc[split_idx:]

print(f"Train: {len(train_weekly)} weeks, Test: {len(test_weekly)} weeks")

X_train_w, y_train_w = train_weekly[feature_cols_weekly], train_weekly[target_col]
X_test_w, y_test_w = test_weekly[feature_cols_weekly], test_weekly[target_col]

### Weekly model: train + evaluate

In [0]:
import mlflow
import mlflow.lightgbm
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error, mean_squared_error

def wape(y_true, y_pred):
    return abs(y_true - y_pred).sum() / abs(y_true).sum()

with mlflow.start_run(run_name="phase1_overall_weekly_netunits"):
    params = {"n_estimators": 200, "learning_rate": 0.05, "max_depth": 6}
    mlflow.log_params(params)

    model_w = lgb.LGBMRegressor(**params)
    model_w.fit(X_train_w, y_train_w)

    preds_w = model_w.predict(X_test_w)

    mae_w = mean_absolute_error(y_test_w, preds_w)
    rmse_w = mean_squared_error(y_test_w, preds_w) ** 0.5
    wape_w = wape(y_test_w, preds_w)

    mlflow.log_metric("mae", mae_w)
    mlflow.log_metric("rmse", rmse_w)
    mlflow.log_metric("wape", wape_w)
    mlflow.lightgbm.log_model(model_w, name="model")

    print(f"Weekly Model — MAE: {mae_w:.2f}   RMSE: {rmse_w:.2f}   WAPE: {wape_w:.3%}")

    baseline_preds_w = X_test_w["rolling_avg_4w"]
    print(f"Weekly Baseline — MAE: {mean_absolute_error(y_test_w, baseline_preds_w):.2f}   WAPE: {wape(y_test_w, baseline_preds_w):.3%}")

In [0]:
plt.figure(figsize=(16, 5))
plt.plot(test_weekly["week_start"], y_test_w.values, label="Actual", marker="o")
plt.plot(test_weekly["week_start"], preds_w, label="Predicted", marker="x")
plt.legend()
plt.title("Weekly Model — Actual vs Predicted (Test Set)")
plt.ylabel("Net Units")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

Weekly: predict next week

In [0]:
X_all_w = model_data_weekly[feature_cols_weekly]
y_all_w = model_data_weekly[target_col]

final_weekly_model = lgb.LGBMRegressor(n_estimators=200, learning_rate=0.05, max_depth=6)
final_weekly_model.fit(X_all_w, y_all_w)

last_week_start = gold_weekly["week_start"].max()
next_week_start = last_week_start + pd.Timedelta(weeks=1)
next_week_end = next_week_start + pd.Timedelta(days=6)

lag_4w_value = gold_weekly[gold_weekly["week_start"] == next_week_start - pd.Timedelta(weeks=4)]["total_units_sold"].values
lag_4w_value = lag_4w_value[0] if len(lag_4w_value) > 0 else None

next_week_features = pd.DataFrame([{
    "week_of_year": next_week_start.isocalendar()[1],
    "month": next_week_start.month,
    "contains_month_end": int(next_week_start.month != next_week_end.month),
    "lag_4w": lag_4w_value,
    "rolling_avg_4w": gold_weekly["total_units_sold"].tail(4).mean(),
}])

next_week_prediction = final_weekly_model.predict(next_week_features[feature_cols_weekly])
print(f"Predicted net units for week starting {next_week_start.date()}: {next_week_prediction[0]:.0f}")

### Monthly model

In [0]:
from prophet import Prophet

monthly_prophet_df = gold_monthly[["month_start", "total_units_sold"]].rename(
    columns={"month_start": "ds", "total_units_sold": "y"}
)

train_monthly = monthly_prophet_df.iloc[:-3]
test_monthly = monthly_prophet_df.iloc[-3:]

m_test = Prophet(yearly_seasonality=True, weekly_seasonality=False)
m_test.fit(train_monthly)

future_test = m_test.make_future_dataframe(periods=3, freq="MS")
forecast_test = m_test.predict(future_test)

test_preds = forecast_test.tail(3)["yhat"].clip(lower=0).values
print(f"Monthly Prophet WAPE: {wape(test_monthly['y'].values, test_preds):.3%}")

In [0]:
plt.figure(figsize=(10, 5))
plt.plot(test_monthly["ds"], test_monthly["y"], label="Actual", marker="o")
plt.plot(test_monthly["ds"], test_preds, label="Predicted", marker="x")
plt.legend()
plt.title("Monthly Prophet — Actual vs Predicted (Last 3 Months, Backtest)")
plt.ylabel("Net Units")
plt.tight_layout()
plt.show()

### Monthly: 3-month future forecast

In [0]:
m_final = Prophet(yearly_seasonality=True, weekly_seasonality=False)
m_final.fit(monthly_prophet_df)

future_final = m_final.make_future_dataframe(periods=3, freq="MS")
forecast_final = m_final.predict(future_final)
forecast_final[["yhat", "yhat_lower", "yhat_upper"]] = forecast_final[["yhat", "yhat_lower", "yhat_upper"]].clip(lower=0)

print(forecast_final[["ds", "yhat", "yhat_lower", "yhat_upper"]].tail(3))

In [0]:
fig = m_final.plot(forecast_final)
plt.title("Monthly Forecast — Full History + Next 3 Months")
plt.ylabel("Net Units")
plt.tight_layout()
plt.show()

In [0]:
naive_monthly_preds = train_monthly["y"].tail(3).mean()  # simple: predict last 3 months' average, repeated
naive_preds_array = [naive_monthly_preds] * 3

print(f"Monthly Naive Baseline WAPE: {wape(test_monthly['y'].values, naive_preds_array):.3%}")

In [0]:
brand_volume = silver.groupby("brand_code").agg(
    total_net_units=("net_units", "sum"),
    row_count=("net_units", "count"),
    date_range_start=("posting_date", "min"),
    date_range_end=("posting_date", "max")
).reset_index()
print(brand_volume)

Brand-wise gold dataset

In [0]:
from src.transform.build_gold_features import build_gold_brand_weekly

silver_trimmed = silver[silver["posting_date"] <= pd.Timestamp("2026-06-12")].copy()

gold_brand_weekly = build_gold_brand_weekly(silver_trimmed)
print(gold_brand_weekly.shape)
gold_brand_weekly.tail(10)

Train the model

In [0]:
gold_brand_weekly["brand_code"] = gold_brand_weekly["brand_code"].astype("category")

feature_cols_brand = ["week_of_year", "month", "contains_month_end", "lag_4w", "rolling_avg_4w", "brand_code"]
target_col = "total_units_sold"

model_data_brand = gold_brand_weekly.dropna(subset=["lag_4w", "rolling_avg_4w", target_col]).copy()
model_data_brand = model_data_brand.sort_values("week_start")

split_idx = int(len(model_data_brand) * 0.8)
train_brand = model_data_brand.iloc[:split_idx]
test_brand = model_data_brand.iloc[split_idx:]

print(f"Train: {len(train_brand)} rows, Test: {len(test_brand)} rows")

X_train_b, y_train_b = train_brand[feature_cols_brand], train_brand[target_col]
X_test_b, y_test_b = test_brand[feature_cols_brand], test_brand[target_col]

Train and evaluate

In [0]:
with mlflow.start_run(run_name="phase2_brand_weekly"):
    params = {"n_estimators": 200, "learning_rate": 0.05, "max_depth": 6}
    mlflow.log_params(params)

    model_brand = lgb.LGBMRegressor(**params)
    model_brand.fit(X_train_b, y_train_b, categorical_feature=["brand_code"])

    preds_b = model_brand.predict(X_test_b)

    overall_wape = wape(y_test_b, preds_b)
    mlflow.log_metric("wape", overall_wape)
    mlflow.lightgbm.log_model(model_brand, name="model")

    print(f"Overall brand-model WAPE: {overall_wape:.3%}")

    test_brand_results = test_brand.copy()
    test_brand_results["prediction"] = preds_b
    for brand in test_brand_results["brand_code"].unique():
        subset = test_brand_results[test_brand_results["brand_code"] == brand]
        brand_wape = wape(subset["total_units_sold"], subset["prediction"])
        baseline_wape = wape(subset["total_units_sold"], subset["rolling_avg_4w"])
        mlflow.log_metric(f"wape_{brand}", brand_wape)
        print(f"{brand} — Model WAPE: {brand_wape:.3%}   Baseline WAPE: {baseline_wape:.3%}   ({len(subset)} weeks)")

In [0]:
dagenite_weekly = gold_brand_weekly[gold_brand_weekly["brand_code"] == "DAGENITE"]
print(dagenite_weekly["total_units_sold"].describe())

In [0]:
final_brand_model = lgb.LGBMRegressor(n_estimators=200, learning_rate=0.05, max_depth=6)
X_all_b = model_data_brand[feature_cols_brand]
y_all_b = model_data_brand[target_col]
final_brand_model.fit(X_all_b, y_all_b, categorical_feature=["brand_code"])

last_week_start = gold_brand_weekly["week_start"].max()
next_week_start = last_week_start + pd.Timedelta(weeks=1)
next_week_end = next_week_start + pd.Timedelta(days=6)

next_week_rows = []
for brand in ["EXIDE", "DAGENITE"]:
    brand_hist = gold_brand_weekly[gold_brand_weekly["brand_code"] == brand]
    lag_val = brand_hist[brand_hist["week_start"] == next_week_start - pd.Timedelta(weeks=4)]["total_units_sold"].values
    lag_val = lag_val[0] if len(lag_val) > 0 else None
    rolling_val = brand_hist["total_units_sold"].tail(4).mean()

    next_week_rows.append({
        "week_of_year": next_week_start.isocalendar()[1],
        "month": next_week_start.month,
        "contains_month_end": int(next_week_start.month != next_week_end.month),
        "lag_4w": lag_val,
        "rolling_avg_4w": rolling_val,
        "brand_code": brand,
    })

next_week_brand_df = pd.DataFrame(next_week_rows)
next_week_brand_df["brand_code"] = next_week_brand_df["brand_code"].astype("category")

predictions = final_brand_model.predict(next_week_brand_df[feature_cols_brand])
for brand, pred in zip(next_week_brand_df["brand_code"], predictions):
    print(f"{brand}: predicted {pred:.0f} net units for week starting {next_week_start.date()}")

Brand-wise monthly

In [0]:
from src.transform.build_gold_features import build_gold_brand_monthly

monthly_cutoff = pd.Timestamp("2026-05-31")
silver_trimmed_monthly = silver[silver["posting_date"] <= monthly_cutoff].copy()

gold_brand_monthly = build_gold_brand_monthly(silver_trimmed_monthly)
print(gold_brand_monthly.tail(10))

Model Train

In [0]:
brand_forecast_results = {}

for brand in ["EXIDE", "DAGENITE"]:
    brand_df = gold_brand_monthly[gold_brand_monthly["brand_code"] == brand][["month_start", "total_units_sold"]]
    brand_df = brand_df.rename(columns={"month_start": "ds", "total_units_sold": "y"})

    train_b = brand_df.iloc[:-3]
    test_b = brand_df.iloc[-3:]

    m_test = Prophet(yearly_seasonality=True, weekly_seasonality=False)
    m_test.fit(train_b)

    future_test = m_test.make_future_dataframe(periods=3, freq="MS")
    forecast_test = m_test.predict(future_test)
    test_preds = forecast_test.tail(3)["yhat"].clip(lower=0).values

    model_wape = wape(test_b["y"].values, test_preds)
    naive_pred = train_b["y"].tail(3).mean()
    baseline_wape = wape(test_b["y"].values, [naive_pred] * 3)

    print(f"{brand} — Model WAPE: {model_wape:.3%}   Naive Baseline WAPE: {baseline_wape:.3%}")

    brand_forecast_results[brand] = {"train": train_b, "test": test_b, "full": brand_df}

3 months forecast

In [0]:
for brand, data in brand_forecast_results.items():
    m_final = Prophet(yearly_seasonality=True, weekly_seasonality=False)
    m_final.fit(data["full"])

    future_final = m_final.make_future_dataframe(periods=3, freq="MS")
    forecast_final = m_final.predict(future_final)
    forecast_final[["yhat", "yhat_lower", "yhat_upper"]] = forecast_final[["yhat", "yhat_lower", "yhat_upper"]].clip(lower=0)

    print(f"\n{brand} — 3-month forecast:")
    print(forecast_final[["ds", "yhat", "yhat_lower", "yhat_upper"]].tail(3))

In [0]:
brand_df_dag = gold_brand_monthly[gold_brand_monthly["brand_code"] == "DAGENITE"][["month_start", "total_units_sold"]]
brand_df_dag = brand_df_dag.rename(columns={"month_start": "ds", "total_units_sold": "y"})

train_dag = brand_df_dag.iloc[:-3]
test_dag = brand_df_dag.iloc[-3:]

m_test_dag = Prophet(yearly_seasonality=False, weekly_seasonality=False)
m_test_dag.fit(train_dag)

future_test_dag = m_test_dag.make_future_dataframe(periods=3, freq="MS")
forecast_test_dag = m_test_dag.predict(future_test_dag)
test_preds_dag = forecast_test_dag.tail(3)["yhat"].clip(lower=0).values

print(f"DAGENITE (no yearly seasonality) WAPE: {wape(test_dag['y'].values, test_preds_dag):.3%}")

In [0]:
# DAGENITE monthly forecast: use naive baseline (3-month rolling average) —
# Prophet does not outperform this for DAGENITE (49.2%/41.5% WAPE vs 38.7% baseline)
dagenite_monthly_forecast = gold_brand_monthly[
    gold_brand_monthly["brand_code"] == "DAGENITE"
]["total_units_sold"].tail(3).mean()

print(f"DAGENITE naive 3-month forecast (flat, repeated): {dagenite_monthly_forecast:.0f} units/month")

Vehicle types

In [0]:
vehicle_volume = silver.groupby("vehicle_type").agg(
    total_net_units=("net_units", "sum"),
    row_count=("net_units", "count"),
    date_range_start=("posting_date", "min"),
    date_range_end=("posting_date", "max")
).reset_index().sort_values("total_net_units", ascending=False)
print(vehicle_volume)

In [0]:
print(f"Null vehicle_type: {silver['vehicle_type'].isna().sum()}")
print(f"Empty string vehicle_type: {(silver['vehicle_type'] == '').sum()}")

In [0]:
cutoff_date = pd.Timestamp("2026-06-12")

vehicle_last_sale = silver.groupby("vehicle_type")["posting_date"].max().reset_index()
vehicle_last_sale.columns = ["vehicle_type", "last_sale_date"]

# Consider a vehicle type "active" if it had sales within the last 6 months of the cutoff
recency_threshold = cutoff_date - pd.Timedelta(days=180)
vehicle_last_sale["is_active"] = vehicle_last_sale["last_sale_date"] >= recency_threshold

print(vehicle_last_sale.sort_values("last_sale_date"))

vehicle_types_to_forecast = vehicle_last_sale[vehicle_last_sale["is_active"]]["vehicle_type"].tolist()
print(f"\nVehicle types to forecast: {vehicle_types_to_forecast}")

In [0]:
brand_by_vehicle = silver.groupby(["vehicle_type", "brand_code"]).agg(
    total_net_units=("net_units", "sum"),
    row_count=("net_units", "count")
).reset_index()

# Pivot for a clean side-by-side view
pivot = brand_by_vehicle.pivot(index="vehicle_type", columns="brand_code", values="total_net_units").fillna(0)
pivot["total"] = pivot.sum(axis=1)
pivot = pivot.sort_values("total", ascending=False)

print(pivot)

In [0]:
pivot_pct = pivot.drop(columns="total").div(pivot.drop(columns="total").sum(axis=1), axis=0) * 100
print(pivot_pct.round(1))

In [0]:
import matplotlib.pyplot as plt

pivot.drop(columns="total").plot(kind="bar", stacked=True, figsize=(12, 6))
plt.title("Battery Units Sold by Vehicle Type and Brand")
plt.ylabel("Net Units")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

Weekly for vehicle types

In [0]:
from src.transform.build_gold_features import build_gold_vehicle_weekly

silver_vehicles = silver[silver["vehicle_type"].isin(vehicle_types_to_forecast)].copy()
silver_trimmed_vehicles = silver_vehicles[silver_vehicles["posting_date"] <= pd.Timestamp("2026-06-12")].copy()

gold_vehicle_weekly = build_gold_vehicle_weekly(silver_trimmed_vehicles)
print(gold_vehicle_weekly.shape)
gold_vehicle_weekly.tail(10)

Train

In [0]:
gold_vehicle_weekly["vehicle_type"] = gold_vehicle_weekly["vehicle_type"].astype("category")

feature_cols_vehicle = ["week_of_year", "month", "contains_month_end", "lag_4w", "rolling_avg_4w", "vehicle_type"]
target_col = "total_units_sold"

model_data_vehicle = gold_vehicle_weekly.dropna(subset=["lag_4w", "rolling_avg_4w", target_col]).copy()
model_data_vehicle = model_data_vehicle.sort_values("week_start")

split_idx = int(len(model_data_vehicle) * 0.8)
train_vehicle = model_data_vehicle.iloc[:split_idx]
test_vehicle = model_data_vehicle.iloc[split_idx:]

print(f"Train: {len(train_vehicle)} rows, Test: {len(test_vehicle)} rows")

X_train_v, y_train_v = train_vehicle[feature_cols_vehicle], train_vehicle[target_col]
X_test_v, y_test_v = test_vehicle[feature_cols_vehicle], test_vehicle[target_col]

model_vehicle = lgb.LGBMRegressor(n_estimators=200, learning_rate=0.05, max_depth=6)
model_vehicle.fit(X_train_v, y_train_v, categorical_feature=["vehicle_type"])

preds_v = model_vehicle.predict(X_test_v)
print(f"\nOverall vehicle-model WAPE: {wape(y_test_v, preds_v):.3%}")

test_vehicle_results = test_vehicle.copy()
test_vehicle_results["prediction"] = preds_v
for vt in test_vehicle_results["vehicle_type"].unique():
    subset = test_vehicle_results[test_vehicle_results["vehicle_type"] == vt]
    v_wape = wape(subset["total_units_sold"], subset["prediction"])
    baseline_wape = wape(subset["total_units_sold"], subset["rolling_avg_4w"])
    print(f"{vt:15s} — Model: {v_wape:.3%}   Baseline: {baseline_wape:.3%}   ({len(subset)} weeks)")

Next-week predictions

In [0]:
vehicle_forecast_method = {
    "LORRY": "model", "CAR": "model", "SUV": "model",
    "THREE WHEELER": "model", "BUS": "model",
    "VAN": "baseline", "TRAILERS": "baseline", "BOATS": "baseline",
}

In [0]:
final_vehicle_model = lgb.LGBMRegressor(n_estimators=200, learning_rate=0.05, max_depth=6)
X_all_v = model_data_vehicle[feature_cols_vehicle]
y_all_v = model_data_vehicle[target_col]
final_vehicle_model.fit(X_all_v, y_all_v, categorical_feature=["vehicle_type"])

last_week_start = gold_vehicle_weekly["week_start"].max()
next_week_start = last_week_start + pd.Timedelta(weeks=1)
next_week_end = next_week_start + pd.Timedelta(days=6)

print(f"Predictions for week starting {next_week_start.date()}:\n")

for vt in vehicle_types_to_forecast:
    vt_hist = gold_vehicle_weekly[gold_vehicle_weekly["vehicle_type"] == vt]

    if vehicle_forecast_method[vt] == "model":
        lag_val = vt_hist[vt_hist["week_start"] == next_week_start - pd.Timedelta(weeks=4)]["total_units_sold"].values
        lag_val = lag_val[0] if len(lag_val) > 0 else None
        rolling_val = vt_hist["total_units_sold"].tail(4).mean()

        row = pd.DataFrame([{
            "week_of_year": next_week_start.isocalendar()[1],
            "month": next_week_start.month,
            "contains_month_end": int(next_week_start.month != next_week_end.month),
            "lag_4w": lag_val,
            "rolling_avg_4w": rolling_val,
            "vehicle_type": vt,
        }])
        row["vehicle_type"] = row["vehicle_type"].astype("category")
        pred = final_vehicle_model.predict(row[feature_cols_vehicle])[0]
        print(f"{vt:15s} (model):    {pred:.0f}")
    else:
        pred = vt_hist["total_units_sold"].tail(4).mean()
        print(f"{vt:15s} (baseline): {pred:.0f}")

In [0]:
vehicle_total = 317 + 1142 + 1215 + 760 + 1691 + 22 + 12 + 19  # sum of all 8 predictions
print(f"Sum of vehicle-type predictions: {vehicle_total}")

In [0]:
# Check: does silver's vehicle_type only ever contain your known 9 values (8 active + INVERTER)?
print(silver["vehicle_type"].value_counts())

# Compare total net_units across all of silver vs silver_vehicles
print(f"Total silver net_units: {silver['net_units'].sum()}")
print(f"Total silver_vehicles net_units: {silver_vehicles['net_units'].sum()}")

In [0]:
# What did vehicle types actually sum to on the most recent known week (not predicted, but real)?
last_known_week = gold_vehicle_weekly["week_start"].max()
actual_vehicle_sum = gold_vehicle_weekly[gold_vehicle_weekly["week_start"] == last_known_week]["total_units_sold"].sum()

actual_overall = gold_weekly[gold_weekly["week_start"] == last_known_week]["total_units_sold"].values
print(f"Vehicle-type sum (actual, last known week): {actual_vehicle_sum}")
print(f"Overall total (actual, last known week): {actual_overall}")

In [0]:
vehicle_predictions = {
    "BUS": 317, "CAR": 1142, "LORRY": 1215, "SUV": 760,
    "THREE WHEELER": 1691, "BOATS": 19, "TRAILERS": 22, "VAN": 12
}

vehicle_sum = sum(vehicle_predictions.values())
overall_prediction = 7432

reconciled = {vt: (val / vehicle_sum) * overall_prediction for vt, val in vehicle_predictions.items()}
for vt, val in sorted(reconciled.items(), key=lambda x: -x[1]):
    print(f"{vt:15s}: {val:.0f}")

In [0]:
# Step 1: Get the overall model's prediction for next week (reuse your existing overall weekly model)
last_week_start_overall = gold_weekly["week_start"].max()
next_week_start = last_week_start_overall + pd.Timedelta(weeks=1)
next_week_end = next_week_start + pd.Timedelta(days=6)

lag_4w_value = gold_weekly[gold_weekly["week_start"] == next_week_start - pd.Timedelta(weeks=4)]["total_units_sold"].values
lag_4w_value = lag_4w_value[0] if len(lag_4w_value) > 0 else None

overall_next_week_features = pd.DataFrame([{
    "week_of_year": next_week_start.isocalendar()[1],
    "month": next_week_start.month,
    "contains_month_end": int(next_week_start.month != next_week_end.month),
    "lag_4w": lag_4w_value,
    "rolling_avg_4w": gold_weekly["total_units_sold"].tail(4).mean(),
}])

overall_prediction = final_weekly_model.predict(overall_next_week_features[feature_cols_weekly])[0]


# Step 2: Get each vehicle type's prediction (model or baseline, per the hybrid approach)
vehicle_predictions = {}

for vt in vehicle_types_to_forecast:
    vt_hist = gold_vehicle_weekly[gold_vehicle_weekly["vehicle_type"] == vt]

    if vehicle_forecast_method[vt] == "model":
        lag_val = vt_hist[vt_hist["week_start"] == next_week_start - pd.Timedelta(weeks=4)]["total_units_sold"].values
        lag_val = lag_val[0] if len(lag_val) > 0 else None
        rolling_val = vt_hist["total_units_sold"].tail(4).mean()

        row = pd.DataFrame([{
            "week_of_year": next_week_start.isocalendar()[1],
            "month": next_week_start.month,
            "contains_month_end": int(next_week_start.month != next_week_end.month),
            "lag_4w": lag_val,
            "rolling_avg_4w": rolling_val,
            "vehicle_type": vt,
        }])
        row["vehicle_type"] = row["vehicle_type"].astype("category")
        vehicle_predictions[vt] = final_vehicle_model.predict(row[feature_cols_vehicle])[0]
    else:
        vehicle_predictions[vt] = vt_hist["total_units_sold"].tail(4).mean()


# Step 3: Reconcile — scale vehicle-type predictions so they sum to the overall prediction
vehicle_sum = sum(vehicle_predictions.values())

reconciled = {vt: (val / vehicle_sum) * overall_prediction for vt, val in vehicle_predictions.items()}

print(f"Overall model prediction for week of {next_week_start.date()}: {overall_prediction:.0f}\n")
print("Reconciled vehicle-type breakdown:")
for vt, val in sorted(reconciled.items(), key=lambda x: -x[1]):
    print(f"  {vt:15s}: {val:.0f}")

print(f"\nSum check: {sum(reconciled.values()):.0f} (should equal {overall_prediction:.0f})")

Vehicle-type monthly

In [0]:
from src.transform.build_gold_features import build_gold_vehicle_monthly

monthly_cutoff = pd.Timestamp("2026-05-31")
silver_trimmed_vehicles_monthly = silver_vehicles[silver_vehicles["posting_date"] <= monthly_cutoff].copy()

gold_vehicle_monthly = build_gold_vehicle_monthly(silver_trimmed_vehicles_monthly)
print(gold_vehicle_monthly.shape)
gold_vehicle_monthly.tail(10)

Backtest Prophet vs naive baseline

In [0]:
vehicle_monthly_method = {}
vehicle_monthly_models = {}
vehicle_monthly_wape = {}

for vt in vehicle_types_to_forecast:
    vt_df = gold_vehicle_monthly[gold_vehicle_monthly["vehicle_type"] == vt][["month_start", "total_units_sold"]]
    vt_df = vt_df.rename(columns={"month_start": "ds", "total_units_sold": "y"})

    if len(vt_df) < 12:  # not enough history to bother with Prophet
        vehicle_monthly_method[vt] = "baseline"
        continue

    train_vt = vt_df.iloc[:-3]
    test_vt = vt_df.iloc[-3:]

    m_test = Prophet(yearly_seasonality=True, weekly_seasonality=False)
    m_test.fit(train_vt)

    future_test = m_test.make_future_dataframe(periods=3, freq="MS")
    forecast_test = m_test.predict(future_test)
    test_preds = forecast_test.tail(3)["yhat"].clip(lower=0).values

    model_wape = wape(test_vt["y"].values, test_preds)
    naive_pred = train_vt["y"].tail(3).mean()
    baseline_wape = wape(test_vt["y"].values, [naive_pred] * 3)

    # Decide the winner dynamically by comparing the two WAPEs
    vehicle_monthly_method[vt] = "model" if model_wape < baseline_wape else "baseline"
    vehicle_monthly_wape[vt] = {"model": model_wape, "baseline": baseline_wape}

    print(f"{vt:15s} — Model WAPE: {model_wape:.3%}   Baseline WAPE: {baseline_wape:.3%}   → using {vehicle_monthly_method[vt]}")

Final 3-month forecast per vehicle type

In [0]:
vehicle_monthly_forecast = {}  # vt -> list of 3 predicted values

for vt in vehicle_types_to_forecast:
    vt_df = gold_vehicle_monthly[gold_vehicle_monthly["vehicle_type"] == vt][["month_start", "total_units_sold"]]
    vt_df = vt_df.rename(columns={"month_start": "ds", "total_units_sold": "y"})

    if vehicle_monthly_method[vt] == "model":
        m_final = Prophet(yearly_seasonality=True, weekly_seasonality=False)
        m_final.fit(vt_df)
        future_final = m_final.make_future_dataframe(periods=3, freq="MS")
        forecast_final = m_final.predict(future_final)
        preds = forecast_final.tail(3)["yhat"].clip(lower=0).values
    else:
        flat_value = vt_df["y"].tail(3).mean()
        preds = [flat_value] * 3

    vehicle_monthly_forecast[vt] = preds
    print(f"{vt:15s}: {[f'{p:.0f}' for p in preds]}")

Overall monthly forecast

In [0]:
# Overall monthly forecast — reuse m_final from your Phase 1 monthly section
overall_monthly_preds = forecast_final.tail(3)["yhat"].clip(lower=0).values if 'm_final' in dir() else None
# NOTE: if the overall m_final was overwritten by the vehicle-type loop above, re-fit it fresh here to be safe:

overall_prophet_df = gold_monthly[["month_start", "total_units_sold"]].rename(columns={"month_start": "ds", "total_units_sold": "y"})
m_overall_final = Prophet(yearly_seasonality=True, weekly_seasonality=False)
m_overall_final.fit(overall_prophet_df)
future_overall = m_overall_final.make_future_dataframe(periods=3, freq="MS")
forecast_overall = m_overall_final.predict(future_overall)
overall_monthly_preds = forecast_overall.tail(3)["yhat"].clip(lower=0).values

months = forecast_overall.tail(3)["ds"].dt.strftime("%Y-%m").tolist()

print("\nReconciled vehicle-type monthly breakdown:\n")
for i, month in enumerate(months):
    month_overall = overall_monthly_preds[i]
    month_vehicle_sum = sum(vehicle_monthly_forecast[vt][i] for vt in vehicle_types_to_forecast)

    print(f"--- {month} (overall: {month_overall:.0f}) ---")
    for vt in sorted(vehicle_types_to_forecast, key=lambda v: -vehicle_monthly_forecast[v][i]):
        raw_val = vehicle_monthly_forecast[vt][i]
        reconciled_val = (raw_val / month_vehicle_sum) * month_overall
        print(f"  {vt:15s}: {reconciled_val:.0f}")
    print()

Per vehicle type × brand combination

In [0]:
cutoff_date = pd.Timestamp("2026-06-12")
recency_threshold = cutoff_date - pd.Timedelta(days=180)

vehicle_last_sale = silver.groupby("vehicle_type")["posting_date"].max().reset_index()
vehicle_last_sale.columns = ["vehicle_type", "last_sale_date"]
vehicle_last_sale["is_active"] = vehicle_last_sale["last_sale_date"] >= recency_threshold

vehicle_types_to_forecast = vehicle_last_sale[vehicle_last_sale["is_active"]]["vehicle_type"].tolist()
print(f"Vehicle types to forecast: {vehicle_types_to_forecast}")

silver_vehicles = silver[silver["vehicle_type"].isin(vehicle_types_to_forecast)].copy()
print(silver_vehicles.shape)

In [0]:
combo_volume = silver_vehicles.groupby(["vehicle_type", "brand_code"]).agg(
    total_net_units=("net_units", "sum"),
    row_count=("net_units", "count"),
    last_sale=("posting_date", "max")
).reset_index().sort_values("total_net_units", ascending=False)

print(combo_volume.to_string())

Build the dataset

In [0]:
from src.transform.build_gold_features import build_gold_vehicle_brand_weekly

silver_trimmed_vb = silver_vehicles[silver_vehicles["posting_date"] <= pd.Timestamp("2026-06-12")].copy()

gold_vehicle_brand_weekly = build_gold_vehicle_brand_weekly(silver_trimmed_vb)
print(gold_vehicle_brand_weekly.shape)

Train

In [0]:
gold_vehicle_brand_weekly["vehicle_type"] = gold_vehicle_brand_weekly["vehicle_type"].astype("category")
gold_vehicle_brand_weekly["brand_code"] = gold_vehicle_brand_weekly["brand_code"].astype("category")

feature_cols_vb = ["week_of_year", "month", "contains_month_end", "lag_4w", "rolling_avg_4w", "vehicle_type", "brand_code"]
target_col = "total_units_sold"

model_data_vb = gold_vehicle_brand_weekly.dropna(subset=["lag_4w", "rolling_avg_4w", target_col]).copy()
model_data_vb = model_data_vb.sort_values("week_start")

split_idx = int(len(model_data_vb) * 0.8)
train_vb = model_data_vb.iloc[:split_idx]
test_vb = model_data_vb.iloc[split_idx:]

X_train_vb, y_train_vb = train_vb[feature_cols_vb], train_vb[target_col]
X_test_vb, y_test_vb = test_vb[feature_cols_vb], test_vb[target_col]

model_vb = lgb.LGBMRegressor(n_estimators=200, learning_rate=0.05, max_depth=6)
model_vb.fit(X_train_vb, y_train_vb, categorical_feature=["vehicle_type", "brand_code"])

preds_vb = model_vb.predict(X_test_vb)
print(f"Overall vehicle+brand model WAPE: {wape(y_test_vb, preds_vb):.3%}")

Evaluate

In [0]:
test_vb_results = test_vb.copy()
test_vb_results["prediction"] = preds_vb

real_combos = model_data_vb[["vehicle_type", "brand_code"]].drop_duplicates().values.tolist()

combo_method = {}
for vt, brand in real_combos:
    subset = test_vb_results[(test_vb_results["vehicle_type"] == vt) & (test_vb_results["brand_code"] == brand)]
    if len(subset) == 0:
        continue
    model_wape = wape(subset["total_units_sold"], subset["prediction"])
    baseline_wape = wape(subset["total_units_sold"], subset["rolling_avg_4w"])
    combo_method[(vt, brand)] = "model" if model_wape < baseline_wape else "baseline"
    print(f"{vt:15s} x {brand:10s} — Model: {model_wape:.3%}   Baseline: {baseline_wape:.3%}   → {combo_method[(vt, brand)]}")

Predict next week

In [0]:
final_vb_model = lgb.LGBMRegressor(n_estimators=200, learning_rate=0.05, max_depth=6)
X_all_vb = model_data_vb[feature_cols_vb]
y_all_vb = model_data_vb[target_col]
final_vb_model.fit(X_all_vb, y_all_vb, categorical_feature=["vehicle_type", "brand_code"])

last_week_start = gold_vehicle_brand_weekly["week_start"].max()
next_week_start = last_week_start + pd.Timedelta(weeks=1)
next_week_end = next_week_start + pd.Timedelta(days=6)

for vt, brand in real_combos:
    combo_hist = gold_vehicle_brand_weekly[
        (gold_vehicle_brand_weekly["vehicle_type"] == vt) & (gold_vehicle_brand_weekly["brand_code"] == brand)
    ]

    if combo_method.get((vt, brand)) == "model":
        lag_val = combo_hist[combo_hist["week_start"] == next_week_start - pd.Timedelta(weeks=4)]["total_units_sold"].values
        lag_val = lag_val[0] if len(lag_val) > 0 else None
        rolling_val = combo_hist["total_units_sold"].tail(4).mean()

        row = pd.DataFrame([{
            "week_of_year": next_week_start.isocalendar()[1],
            "month": next_week_start.month,
            "contains_month_end": int(next_week_start.month != next_week_end.month),
            "lag_4w": lag_val,
            "rolling_avg_4w": rolling_val,
            "vehicle_type": vt,
            "brand_code": brand,
        }])
        row["vehicle_type"] = row["vehicle_type"].astype("category")
        row["brand_code"] = row["brand_code"].astype("category")
        pred = final_vb_model.predict(row[feature_cols_vb])[0]
    else:
        pred = combo_hist["total_units_sold"].tail(4).mean()

    if vt == "THREE WHEELER":
        print(f"{vt} x {brand}: {pred:.0f}")

In [0]:
last_week_start = gold_vehicle_brand_weekly["week_start"].max()
next_week_start = last_week_start + pd.Timedelta(weeks=1)
next_week_end = next_week_start + pd.Timedelta(days=6)

real_combos = model_data_vb[["vehicle_type", "brand_code"]].drop_duplicates().values.tolist()

raw_vb_predictions = {}

for vt, brand in real_combos:
    combo_hist = gold_vehicle_brand_weekly[
        (gold_vehicle_brand_weekly["vehicle_type"] == vt) & (gold_vehicle_brand_weekly["brand_code"] == brand)
    ]

    if combo_method.get((vt, brand)) == "model":
        lag_val = combo_hist[combo_hist["week_start"] == next_week_start - pd.Timedelta(weeks=4)]["total_units_sold"].values
        lag_val = lag_val[0] if len(lag_val) > 0 else None
        rolling_val = combo_hist["total_units_sold"].tail(4).mean()

        row = pd.DataFrame([{
            "week_of_year": next_week_start.isocalendar()[1],
            "month": next_week_start.month,
            "contains_month_end": int(next_week_start.month != next_week_end.month),
            "lag_4w": lag_val,
            "rolling_avg_4w": rolling_val,
            "vehicle_type": vt,
            "brand_code": brand,
        }])
        row["vehicle_type"] = row["vehicle_type"].astype("category")
        row["brand_code"] = row["brand_code"].astype("category")
        pred = final_vb_model.predict(row[feature_cols_vb])[0]
    else:
        pred = combo_hist["total_units_sold"].tail(4).mean()

    raw_vb_predictions[(vt, brand)] = max(pred, 0)  # clip negatives, shouldn't happen but safe

In [0]:
# 'reconciled' from your earlier vehicle-type step maps vehicle_type -> reconciled value
# Recompute fresh here to make sure it's not stale
vehicle_type_targets = reconciled  # {vt: reconciled_value}

In [0]:
final_vb_reconciled = {}

for vt in vehicle_types_to_forecast:
    combos_for_vt = [(v, b) for (v, b) in raw_vb_predictions if v == vt]
    raw_sum_for_vt = sum(raw_vb_predictions[c] for c in combos_for_vt)
    target_for_vt = vehicle_type_targets.get(vt, 0)

    if raw_sum_for_vt == 0:
        continue

    for combo in combos_for_vt:
        share = raw_vb_predictions[combo] / raw_sum_for_vt
        final_vb_reconciled[combo] = share * target_for_vt

print("Reconciled vehicle x brand breakdown:\n")
for vt in vehicle_types_to_forecast:
    combos_for_vt = sorted([(v, b) for (v, b) in final_vb_reconciled if v == vt], key=lambda c: -final_vb_reconciled[c])
    vt_total = sum(final_vb_reconciled[c] for c in combos_for_vt)
    print(f"{vt} (total: {vt_total:.0f}):")
    for (v, b) in combos_for_vt:
        print(f"    {b:10s}: {final_vb_reconciled[(v, b)]:.0f}")

In [0]:
# Each vehicle type's brand breakdown should sum to its own reconciled total
for vt in vehicle_types_to_forecast:
    combos_for_vt = [(v, b) for (v, b) in final_vb_reconciled if v == vt]
    print(f"{vt}: sum={sum(final_vb_reconciled[c] for c in combos_for_vt):.0f}  target={vehicle_type_targets.get(vt, 0):.0f}")

# Grand total should equal the overall model's prediction
print(f"\nGrand total: {sum(final_vb_reconciled.values()):.0f}  (should equal overall_prediction: {overall_prediction:.0f})")

Vehicle×brand monthly

In [0]:
from src.transform.build_gold_features import build_gold_vehicle_brand_monthly

monthly_cutoff = pd.Timestamp("2026-05-31")
silver_vehicles_monthly = silver_vehicles[silver_vehicles["posting_date"] <= monthly_cutoff].copy()

gold_vehicle_brand_monthly = build_gold_vehicle_brand_monthly(silver_vehicles_monthly)
print(gold_vehicle_brand_monthly.shape)

Prophet vs naive

In [0]:
real_combos_monthly = gold_vehicle_brand_monthly[["vehicle_type", "brand_code"]].drop_duplicates().values.tolist()

vb_monthly_method = {}
vb_monthly_forecast = {}  # (vt, brand) -> list of 3 predicted values

for vt, brand in real_combos_monthly:
    combo_df = gold_vehicle_brand_monthly[
        (gold_vehicle_brand_monthly["vehicle_type"] == vt) & (gold_vehicle_brand_monthly["brand_code"] == brand)
    ][["month_start", "total_units_sold"]].rename(columns={"month_start": "ds", "total_units_sold": "y"})

    if len(combo_df) < 15 or combo_df["y"].tail(12).sum() == 0:
        # Not enough history, or effectively no recent activity — use naive baseline directly
        vb_monthly_method[(vt, brand)] = "baseline"
        flat_value = max(combo_df["y"].tail(3).mean(), 0)
        vb_monthly_forecast[(vt, brand)] = [flat_value] * 3
        continue

    train_c = combo_df.iloc[:-3]
    test_c = combo_df.iloc[-3:]

    try:
        m_test = Prophet(yearly_seasonality=True, weekly_seasonality=False)
        m_test.fit(train_c)
        future_test = m_test.make_future_dataframe(periods=3, freq="MS")
        forecast_test = m_test.predict(future_test)
        test_preds = forecast_test.tail(3)["yhat"].clip(lower=0).values

        model_wape = wape(test_c["y"].values, test_preds)
        naive_pred = train_c["y"].tail(3).mean()
        baseline_wape = wape(test_c["y"].values, [naive_pred] * 3)

        method = "model" if model_wape < baseline_wape else "baseline"
    except Exception:
        method = "baseline"

    vb_monthly_method[(vt, brand)] = method
    print(f"{vt:15s} x {brand:10s} → {method}")

    # Final 3-month forecast using the winning method, refit on all data
    if method == "model":
        m_final = Prophet(yearly_seasonality=True, weekly_seasonality=False)
        m_final.fit(combo_df)
        future_final = m_final.make_future_dataframe(periods=3, freq="MS")
        forecast_final = m_final.predict(future_final)
        preds = forecast_final.tail(3)["yhat"].clip(lower=0).values
    else:
        flat_value = max(combo_df["y"].tail(3).mean(), 0)
        preds = [flat_value] * 3

    vb_monthly_forecast[(vt, brand)] = preds

In [0]:
vehicle_monthly_targets = {}  # month_str -> {vt: reconciled_value}

for i in range(3):
    month_overall = overall_monthly_preds[i]
    month_vehicle_sum = sum(vehicle_monthly_forecast[vt][i] for vt in vehicle_types_to_forecast)
    month_str = months[i]

    vehicle_monthly_targets[month_str] = {}
    for vt in vehicle_types_to_forecast:
        raw_val = vehicle_monthly_forecast[vt][i]
        vehicle_monthly_targets[month_str][vt] = (raw_val / month_vehicle_sum) * month_overall

Monthly forecast

In [0]:
final_vb_monthly_reconciled = {}

for i, month_str in enumerate(months):
    final_vb_monthly_reconciled[month_str] = {}

    for vt in vehicle_types_to_forecast:
        combos_for_vt = [(v, b) for (v, b) in vb_monthly_forecast if v == vt]
        raw_sum_for_vt = sum(vb_monthly_forecast[c][i] for c in combos_for_vt)
        target_for_vt = vehicle_monthly_targets[month_str].get(vt, 0)

        if raw_sum_for_vt == 0:
            continue

        for combo in combos_for_vt:
            share = vb_monthly_forecast[combo][i] / raw_sum_for_vt
            final_vb_monthly_reconciled[month_str][combo] = share * target_for_vt

print("Reconciled vehicle x brand monthly breakdown:\n")
for month_str in months:
    print(f"--- {month_str} ---")
    for vt in vehicle_types_to_forecast:
        combos_for_vt = sorted(
            [(v, b) for (v, b) in final_vb_monthly_reconciled[month_str] if v == vt],
            key=lambda c: -final_vb_monthly_reconciled[month_str][c]
        )
        vt_total = sum(final_vb_monthly_reconciled[month_str][c] for c in combos_for_vt)
        print(f"  {vt} (total: {vt_total:.0f}):")
        for combo in combos_for_vt:
            print(f"      {combo[1]:10s}: {final_vb_monthly_reconciled[month_str][combo]:.0f}")
    print()

Active locations

In [0]:
location_summary = silver.groupby(["location_code", "location_description"]).agg(
    total_net_units=("net_units", "sum"),
    row_count=("net_units", "count")
).reset_index().sort_values("total_net_units", ascending=False)

print(location_summary.to_string())

In [0]:
location_stats = silver.groupby(["location_code", "location_description"]).agg(
    total_net_units=("net_units", "sum"),
    last_sale_date=("posting_date", "max")
).reset_index()

cutoff_date = pd.Timestamp("2026-06-12")
recency_threshold = cutoff_date - pd.Timedelta(days=180)

location_stats["is_active"] = (
    (location_stats["last_sale_date"] >= recency_threshold) &
    (location_stats["total_net_units"] >= 1000)
)

print(location_stats.sort_values("total_net_units", ascending=False))

locations_to_forecast = location_stats[location_stats["is_active"]]["location_code"].tolist()
print(f"\nLocations to forecast: {locations_to_forecast}")

Train

In [0]:
from src.transform.build_gold_features import build_gold_location_weekly

silver_locations = silver[silver["location_code"].isin(locations_to_forecast)].copy()
silver_trimmed_locations = silver_locations[silver_locations["posting_date"] <= pd.Timestamp("2026-06-12")].copy()

gold_location_weekly = build_gold_location_weekly(silver_trimmed_locations)
gold_location_weekly["location_code"] = gold_location_weekly["location_code"].astype("category")

feature_cols_location = ["week_of_year", "month", "contains_month_end", "lag_4w", "rolling_avg_4w", "location_code"]
target_col = "total_units_sold"

model_data_location = gold_location_weekly.dropna(subset=["lag_4w", "rolling_avg_4w", target_col]).copy()
model_data_location = model_data_location.sort_values("week_start")

split_idx = int(len(model_data_location) * 0.8)
train_location = model_data_location.iloc[:split_idx]
test_location = model_data_location.iloc[split_idx:]

X_train_l, y_train_l = train_location[feature_cols_location], train_location[target_col]
X_test_l, y_test_l = test_location[feature_cols_location], test_location[target_col]

model_location = lgb.LGBMRegressor(n_estimators=200, learning_rate=0.05, max_depth=6)
model_location.fit(X_train_l, y_train_l, categorical_feature=["location_code"])

preds_l = model_location.predict(X_test_l)
print(f"Overall location-model WAPE: {wape(y_test_l, preds_l):.3%}\n")

test_location_results = test_location.copy()
test_location_results["prediction"] = preds_l

location_method = {}
for loc in test_location_results["location_code"].unique():
    subset = test_location_results[test_location_results["location_code"] == loc]
    model_wape = wape(subset["total_units_sold"], subset["prediction"])
    baseline_wape = wape(subset["total_units_sold"], subset["rolling_avg_4w"])
    location_method[loc] = "model" if model_wape < baseline_wape else "baseline"
    print(f"{loc:15s} — Model: {model_wape:.3%}   Baseline: {baseline_wape:.3%}   → {location_method[loc]}")

Weekly Prediction

In [0]:
final_location_model = lgb.LGBMRegressor(n_estimators=200, learning_rate=0.05, max_depth=6)
X_all_l = model_data_location[feature_cols_location]
y_all_l = model_data_location[target_col]
final_location_model.fit(X_all_l, y_all_l, categorical_feature=["location_code"])

last_week_start = gold_location_weekly["week_start"].max()
next_week_start = last_week_start + pd.Timedelta(weeks=1)
next_week_end = next_week_start + pd.Timedelta(days=6)

print(f"Predictions for week starting {next_week_start.date()}:\n")

location_predictions = {}
for loc in locations_to_forecast:
    loc_hist = gold_location_weekly[gold_location_weekly["location_code"] == loc]

    if location_method.get(loc) == "model":
        lag_val = loc_hist[loc_hist["week_start"] == next_week_start - pd.Timedelta(weeks=4)]["total_units_sold"].values
        lag_val = lag_val[0] if len(lag_val) > 0 else None
        rolling_val = loc_hist["total_units_sold"].tail(4).mean()

        row = pd.DataFrame([{
            "week_of_year": next_week_start.isocalendar()[1],
            "month": next_week_start.month,
            "contains_month_end": int(next_week_start.month != next_week_end.month),
            "lag_4w": lag_val,
            "rolling_avg_4w": rolling_val,
            "location_code": loc,
        }])
        row["location_code"] = row["location_code"].astype("category")
        pred = final_location_model.predict(row[feature_cols_location])[0]
    else:
        pred = loc_hist["total_units_sold"].tail(4).mean()

    location_predictions[loc] = max(pred, 0)
    print(f"{loc:15s}: {location_predictions[loc]:.0f}")

In [0]:
from src.transform.build_gold_features import build_gold_location_monthly

monthly_cutoff = pd.Timestamp("2026-05-31")
silver_locations_monthly = silver_locations[silver_locations["posting_date"] <= monthly_cutoff].copy()

gold_location_monthly = build_gold_location_monthly(silver_locations_monthly)
print(gold_location_monthly.shape)

In [0]:
location_monthly_forecast = {}

for loc in locations_to_forecast:
    loc_df = gold_location_monthly[gold_location_monthly["location_code"] == loc][["month_start", "total_units_sold"]]
    loc_df = loc_df.rename(columns={"month_start": "ds", "total_units_sold": "y"})

    if len(loc_df) < 15 or loc_df["y"].tail(12).sum() == 0:
        method = "baseline"
    else:
        train_l = loc_df.iloc[:-3]
        test_l = loc_df.iloc[-3:]

        try:
            m_test = Prophet(yearly_seasonality=True, weekly_seasonality=False)
            m_test.fit(train_l)
            future_test = m_test.make_future_dataframe(periods=3, freq="MS")
            forecast_test = m_test.predict(future_test)
            test_preds = forecast_test.tail(3)["yhat"].clip(lower=0).values

            model_wape = wape(test_l["y"].values, test_preds)
            naive_pred = train_l["y"].tail(3).mean()
            baseline_wape = wape(test_l["y"].values, [naive_pred] * 3)

            method = "model" if model_wape < baseline_wape else "baseline"
            print(f"{loc:15s} — Model WAPE: {model_wape:.3%}   Baseline WAPE: {baseline_wape:.3%}   → {method}")
        except Exception:
            method = "baseline"
            print(f"{loc:15s} — Prophet failed, using baseline")

    if method == "model":
        m_final = Prophet(yearly_seasonality=True, weekly_seasonality=False)
        m_final.fit(loc_df)
        future_final = m_final.make_future_dataframe(periods=3, freq="MS")
        forecast_final = m_final.predict(future_final)
        preds = forecast_final.tail(3)["yhat"].clip(lower=0).values
    else:
        flat_value = max(loc_df["y"].tail(3).mean(), 0)
        preds = [flat_value] * 3

    location_monthly_forecast[loc] = preds

print("\n3-month forecast per location:")
for loc, preds in location_monthly_forecast.items():
    print(f"{loc:15s}: {[f'{p:.0f}' for p in preds]}")

Load location hierarchy table

In [0]:
import os
import pandas as pd

def find_repo_root(marker="requirements.txt"):
    path = os.getcwd()
    while not os.path.exists(os.path.join(path, marker)):
        parent = os.path.dirname(path)
        if parent == path:
            raise FileNotFoundError("Could not find repo root")
        path = parent
    return path

REPO_ROOT = find_repo_root()
location_hierarchy = pd.read_parquet(os.path.join(REPO_ROOT, "data", "location_hierarchy.parquet"))

print(location_hierarchy.shape)
location_hierarchy.head()

In [0]:
import io

buffer = io.BytesIO()
location_hierarchy.to_parquet(buffer, index=False)
buffer.seek(0)

blob_client = blob_service.get_blob_client(container="silver", blob="live/battery/forecast/reference/location_hierarchy.parquet")
blob_client.upload_blob(buffer, overwrite=True)

print("Saved to silver/live/battery/reference/location_hierarchy.parquet in blob storage")

City-matching

In [0]:
from rapidfuzz import process, fuzz

city_names = location_hierarchy["city_name"].unique().tolist()
city_names_upper = [c.upper() for c in city_names]
upper_to_original = dict(zip(city_names_upper, city_names))

def find_city_fuzzy(description: str, threshold: int = 85) -> str:
    words = description.upper().split()
    candidates = words + [" ".join(words[i:i+2]) for i in range(len(words)-1)]

    best_match = None
    best_score = 0
    for candidate in candidates:
        result = process.extractOne(candidate, city_names_upper, scorer=fuzz.ratio)
        if result and result[1] > best_score and result[1] >= threshold:
            best_match = upper_to_original[result[0]]
            best_score = result[1]

    return best_match if best_match else "Others"


location_city_mapping = location_stats[location_stats["is_active"]][["location_code", "location_description"]].copy()
location_city_mapping["matched_city"] = location_city_mapping["location_description"].apply(find_city_fuzzy)

location_city_mapping["match_key"] = location_city_mapping["matched_city"].str.upper()
location_hierarchy["match_key"] = location_hierarchy["city_name"].str.upper()

location_city_mapping = location_city_mapping.merge(
    location_hierarchy[["match_key", "district_name", "province_name"]],
    on="match_key",
    how="left"
)

location_city_mapping["district_name"] = location_city_mapping["district_name"].fillna("Others")
location_city_mapping["province_name"] = location_city_mapping["province_name"].fillna("Others")

print(location_city_mapping[["location_code", "location_description", "matched_city", "district_name", "province_name"]])

Weekly

In [0]:
location_pred_df = pd.DataFrame([
    {"location_code": loc, "predicted_units": val} for loc, val in location_predictions.items()
])

location_pred_with_geo = location_pred_df.merge(
    location_city_mapping[["location_code", "matched_city", "district_name", "province_name"]],
    on="location_code",
    how="left"
)

print("By district:")
print(location_pred_with_geo.groupby("district_name")["predicted_units"].sum().sort_values(ascending=False))

print("\nBy province:")
print(location_pred_with_geo.groupby("province_name")["predicted_units"].sum().sort_values(ascending=False))

Monthly

In [0]:
monthly_rows = []
for loc, preds in location_monthly_forecast.items():
    for i, month_str in enumerate(months):
        monthly_rows.append({"location_code": loc, "month": month_str, "predicted_units": preds[i]})

location_monthly_df = pd.DataFrame(monthly_rows)

location_monthly_with_geo = location_monthly_df.merge(
    location_city_mapping[["location_code", "matched_city", "district_name", "province_name"]],
    on="location_code",
    how="left"
)

print("By district, per month:")
print(location_monthly_with_geo.pivot_table(index="district_name", columns="month", values="predicted_units", aggfunc="sum"))

print("\nBy province, per month:")
print(location_monthly_with_geo.pivot_table(index="province_name", columns="month", values="predicted_units", aggfunc="sum"))

Season blocks across years

In [0]:
gold_monthly["year"] = gold_monthly["month_start"].dt.year

season_by_year = gold_monthly.groupby(["year", "season_block"], observed=True).agg(
    total_units=("total_units_sold", "sum"),
    avg_units=("total_units_sold", "mean")
).reset_index()

print(season_by_year.pivot(index="year", columns="season_block", values="total_units"))

In [0]:
import matplotlib.pyplot as plt

pivot_seasonal = season_by_year.pivot(index="year", columns="season_block", values="total_units")
pivot_seasonal.plot(kind="bar", figsize=(12, 6))
plt.title("Total Units Sold by Season Block, Per Year")
plt.ylabel("Net Units")
plt.tight_layout()
plt.show()

In [0]:
gold_monthly["season_block"] = gold_monthly["season_block"].astype("category")

feature_cols_with_season = ["month_of_year", "quarter", "season_block", "lag_1m", "lag_12m", "rolling_avg_3m"]
feature_cols_without_season = ["month_of_year", "quarter", "lag_1m", "lag_12m", "rolling_avg_3m"]
target_col = "total_units_sold"

model_data_season = gold_monthly.dropna(subset=feature_cols_with_season + [target_col]).copy()
model_data_season = model_data_season.sort_values("month_start")

split_idx = int(len(model_data_season) * 0.8)
train_s = model_data_season.iloc[:split_idx]
test_s = model_data_season.iloc[split_idx:]

for label, cols in [("WITH season_block", feature_cols_with_season), ("WITHOUT season_block", feature_cols_without_season)]:
    X_train_s, y_train_s = train_s[cols], train_s[target_col]
    X_test_s, y_test_s = test_s[cols], test_s[target_col]

    cat_features = ["season_block"] if "season_block" in cols else []
    m = lgb.LGBMRegressor(n_estimators=100, learning_rate=0.05, max_depth=4)
    m.fit(X_train_s, y_train_s, categorical_feature=cat_features)

    preds_s = m.predict(X_test_s)
    print(f"{label} — WAPE: {wape(y_test_s, preds_s):.3%}")

In [0]:
def get_season_block(month_num):
    if month_num <= 4:
        return "first_4mo"
    elif month_num <= 8:
        return "mid_4mo"
    else:
        return "last_4mo"

forecast_with_season = pd.DataFrame({
    "month": months,
    "predicted_units": overall_monthly_preds
})
forecast_with_season["month_num"] = pd.to_datetime(forecast_with_season["month"]).dt.month
forecast_with_season["season_block"] = forecast_with_season["month_num"].apply(get_season_block)

print(forecast_with_season)
print("\nTotal by season block (forecast period):")
print(forecast_with_season.groupby("season_block")["predicted_units"].sum())

Fraud analysis

In [0]:
import matplotlib.pyplot as plt

shipments = silver[silver["documentType"] == "Sales Shipment"].copy()
returns = silver[silver["documentType"] == "Sales Return Receipt"].copy()

shipments["day_of_month"] = shipments["posting_date"].dt.day
returns["day_of_month"] = returns["posting_date"].dt.day

avg_shipments_by_day = shipments.groupby("day_of_month")["gross_units"].sum() / shipments["posting_date"].dt.to_period("M").nunique()
avg_returns_by_day = returns.groupby("day_of_month")["gross_units"].sum() / returns["posting_date"].dt.to_period("M").nunique()

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

axes[0].bar(avg_shipments_by_day.index, avg_shipments_by_day.values, color="steelblue")
axes[0].set_title("Average Shipment Volume by Day of Month")
axes[0].set_ylabel("Avg Units")

axes[1].bar(avg_returns_by_day.index, avg_returns_by_day.values, color="indianred")
axes[1].set_title("Average Return Volume by Day of Month")
axes[1].set_xlabel("Day of Month")
axes[1].set_ylabel("Avg Units")

plt.tight_layout()
plt.show()

In [0]:
early_month_returns = returns[returns["day_of_month"] <= 15]
avg_early_returns_by_day = early_month_returns.groupby("day_of_month")["gross_units"].sum() / early_month_returns["posting_date"].dt.to_period("M").nunique()

plt.figure(figsize=(12, 5))
plt.bar(avg_early_returns_by_day.index, avg_early_returns_by_day.values, color="indianred")
plt.title("Average Return Volume — First 15 Days of Month Only")
plt.xlabel("Day of Month")
plt.ylabel("Avg Units")
plt.tight_layout()
plt.show()

Isolate candidate shipments and returns

In [0]:
shipments = silver[silver["documentType"] == "Sales Shipment"].copy()
returns = silver[silver["documentType"] == "Sales Return Receipt"].copy()

shipments["day_of_month"] = shipments["posting_date"].dt.day
shipments["days_in_month"] = shipments["posting_date"].dt.days_in_month

# Month-end shipments: last 3 days of the month
month_end_shipments = shipments[
    shipments["day_of_month"] > shipments["days_in_month"] - 3
].copy()
month_end_shipments["shipment_month"] = month_end_shipments["posting_date"].dt.to_period("M")
month_end_shipments["target_return_month"] = month_end_shipments["shipment_month"] + 1

# All returns, no day restriction this time
returns["return_month"] = returns["posting_date"].dt.to_period("M")

print(f"Month-end shipment candidates: {len(month_end_shipments)}")
print(f"Total returns (all): {len(returns)}")

In [0]:
matches = month_end_shipments.merge(
    returns,
    left_on=["sales_person_code", "item_no", "target_return_month"],
    right_on=["sales_person_code", "item_no", "return_month"],
    suffixes=("_shipment", "_return")
)

matches["quantity_diff"] = (matches["gross_units_shipment"] - matches["gross_units_return"]).abs()
matches["quantity_diff_pct"] = matches["quantity_diff"] / matches["gross_units_shipment"]

strong_matches = matches[matches["quantity_diff_pct"] <= 0.20].copy()
strong_matches["days_between"] = (strong_matches["posting_date_return"] - strong_matches["posting_date_shipment"]).dt.days

print(f"Total matched round-trip candidates: {len(matches)}")
print(f"Strong matches (quantity within 20%): {len(strong_matches)}")

In [0]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 5))
plt.hist(strong_matches["days_between"], bins=30, color="indianred")
plt.title("Days Between Month-End Shipment and Matching Return")
plt.xlabel("Days Between")
plt.ylabel("Count of Matches")
plt.tight_layout()
plt.show()

print(strong_matches["days_between"].describe())

In [0]:
salesperson_summary = strong_matches.groupby("sales_person_code").agg(
    round_trip_count=("item_no", "count"),
    total_units_involved=("gross_units_shipment", "sum"),
    total_value_involved=("salesAmountActual_shipment", "sum"),
    unique_months=("shipment_month", "nunique"),
    avg_days_between=("days_between", "mean")
).reset_index().sort_values("round_trip_count", ascending=False)

print(salesperson_summary)

In [0]:
print(silver["sales_person_code"].isna().sum())
print((silver["sales_person_code"] == "").sum())
print(silver["sales_person_code"].value_counts(dropna=False).tail(20))

In [0]:
silver["sales_person_code"] = silver["sales_person_code"].fillna("").astype(str).str.strip()

has_salesperson = silver[silver["sales_person_code"] != ""].copy()
no_salesperson = silver[silver["sales_person_code"] == ""].copy()

print(f"Rows with a salesperson code: {len(has_salesperson)}")
print(f"Rows with NO salesperson code: {len(no_salesperson)}")
print(f"No-salesperson net units: {no_salesperson['net_units'].sum():.0f}")

In [0]:
shipments = has_salesperson[has_salesperson["documentType"] == "Sales Shipment"].copy()
returns = has_salesperson[has_salesperson["documentType"] == "Sales Return Receipt"].copy()

shipments["day_of_month"] = shipments["posting_date"].dt.day
shipments["days_in_month"] = shipments["posting_date"].dt.days_in_month
month_end_shipments = shipments[shipments["day_of_month"] > shipments["days_in_month"] - 3].copy()
month_end_shipments["shipment_month"] = month_end_shipments["posting_date"].dt.to_period("M")
month_end_shipments["target_return_month"] = month_end_shipments["shipment_month"] + 1

returns["return_month"] = returns["posting_date"].dt.to_period("M")

def match_group(ship_group, ret_group, tolerance=0.20):
    used = set()
    pairs = []
    for _, ship in ship_group.sort_values("gross_units", ascending=False).iterrows():
        avail = ret_group[~ret_group.index.isin(used)]
        if avail.empty:
            continue
        qty_diff = (avail["gross_units"] - ship["gross_units"]).abs()
        best_idx = qty_diff.idxmin()
        if qty_diff.loc[best_idx] / ship["gross_units"] <= tolerance:
            best = avail.loc[best_idx]
            pairs.append({
                "sales_person_code": ship["sales_person_code"],
                "item_no": ship["item_no"],
                "shipment_date": ship["posting_date"],
                "shipment_units": ship["gross_units"],
                "shipment_value": ship["salesAmountActual"],
                "return_date": best["posting_date"],
                "return_units": best["gross_units"],
                "days_between": (best["posting_date"] - ship["posting_date"]).days,
            })
            used.add(best_idx)
    return pairs

all_pairs = []
grouped_ship = month_end_shipments.groupby(["sales_person_code", "item_no", "target_return_month"])

for (sp, item, target_month), ship_group in grouped_ship:
    ret_group = returns[
        (returns["sales_person_code"] == sp) &
        (returns["item_no"] == item) &
        (returns["return_month"] == target_month)
    ]
    if len(ret_group) == 0:
        continue
    all_pairs.extend(match_group(ship_group, ret_group))

matches_df = pd.DataFrame(all_pairs)
print(f"Total genuine one-to-one matches: {len(matches_df)}")
print(f"Total returns available to match against: {len(returns)}")

In [0]:
salesperson_summary = matches_df.groupby("sales_person_code").agg(
    round_trip_count=("item_no", "count"),
    total_units_involved=("shipment_units", "sum"),
    total_value_involved=("shipment_value", "sum"),
    avg_days_between=("days_between", "mean")
).reset_index().sort_values("round_trip_count", ascending=False)

print(salesperson_summary)

In [0]:
total_month_end_shipments_per_sp = month_end_shipments.groupby("sales_person_code").size()
salesperson_summary = salesperson_summary.merge(
    total_month_end_shipments_per_sp.rename("total_month_end_shipments"),
    left_on="sales_person_code", right_index=True, how="left"
)
salesperson_summary["round_trip_rate"] = salesperson_summary["round_trip_count"] / salesperson_summary["total_month_end_shipments"]
print(salesperson_summary.sort_values("round_trip_rate", ascending=False)[
    ["sales_person_code", "round_trip_count", "total_month_end_shipments", "round_trip_rate", "total_value_involved"]
])

In [0]:
kam_months = matches_df[matches_df["sales_person_code"] == "KAM-001889"]["shipment_date"].dt.to_period("M").value_counts().sort_index()
print(kam_months)

tmw_months = matches_df[matches_df["sales_person_code"] == "TMW-003149"]["shipment_date"].dt.to_period("M").value_counts().sort_index()
print(tmw_months)

In [0]:
print(matches_df[matches_df["sales_person_code"].isin(["WR", "RNA"])][["item_no", "shipment_value", "shipment_units"]].describe())